In [ ]:
# Generate and plot data

# -------------------------------
# Define your custom path to the 'functions' module
# -------------------------------
import sys
# Replace the path below with your own path to the 'functions' directory
sys.path.append("/path/to/your/functions/")  
print(sys.path)

# Import simulation function
from neurIPS.functions.data_simulation import data_simulation1

# -------------------------------
# Set random seeds for reproducibility
# -------------------------------
seed_value = 1234

import os
os.environ['PYTHONHASHSEED'] = str(seed_value)

import random
random.seed(seed_value)

import numpy as np
np.random.seed(seed_value)

# -------------------------------
# Simulate data
# -------------------------------
data_exp, data_obs = data_simulation1(1000, beta0=-3.0, beta1=-3.0)
data_full = np.r_[data_exp, data_obs]

# -------------------------------
# Model for the probability of participation
# -------------------------------
from sklearn.linear_model import LogisticRegression

logr = LogisticRegression()
logr.fit(data_full[:,1].reshape(-1,1), data_full[:,0])
prob = logr.predict_proba(data_full[:,1].reshape(-1,1))
prob_part = prob[:,1]

# Add inverse probability weights
data_full = np.c_[data_full, 1/prob_part]

plt.scatter(data_full[data_full[:,0]==1,1], data_full[data_full[:,0]==1,4])
plt.xlabel("X")
plt.ylabel("W = 1/Pr[S=1]")

# -------------------------------
# Tune rho
# -------------------------------
from sklearn.model_selection import KFold
from neurIPS.functions.model_training import ICM
from neurIPS.functions.weightedMSE import weighted_mean_squared_error

# Range of rho values to search
rho_values = np.round(np.arange(0, 1.05, 0.1), 1)

num_folds = 5  # 5-fold CV

min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True, random_state=seed_value)

for rho in rho_values:
    avg_wmse = 0
    wmse_per_fold = []
    
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        val_data = val_data[val_data[:,0]==1,:]
        X_val = val_data[:,1]
        T_val = val_data[:,2]
        Y_val = val_data[:,3]
        w = val_data[:,4]

        m0, m1 = ICM(
            X_E=train_data[train_data[:,0]==1,1], 
            X_O=train_data[train_data[:,0]==0,1], 
            T_E=train_data[train_data[:,0]==1,2], 
            T_O=train_data[train_data[:,0]==0,2], 
            Y_E=train_data[train_data[:,0]==1,3], 
            Y_O=train_data[train_data[:,0]==0,3],  
            r=2, ID=1, AD=0, rho=rho
        )

        predY0_exp = m0.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape[0])])
        predY1_exp = m1.predict_noiseless(np.c_[np.vstack(val_data[:,1]), np.zeros(val_data[:,1].shape[0])])
        
        wmse = weighted_mean_squared_error(weight=w, t=T_val, y_true=Y_val, 
                                           y_pred0=predY0_exp[0], y_pred1=predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)
    
    avg_wmse /= num_folds
    wmse_per_rho[rho] = wmse_per_fold

    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best Rho: {best_rho}, Minimum Average WMSE: {min_avg_wmse}")

# Save best rho
import pandas as pd
sim1_best_rho_df = pd.DataFrame([best_rho], columns=['best_rho'])
sim1_best_rho_df.to_csv('sim1_best_rho.csv', index=False)


# -------------------------------
# Train final models with best rho
# -------------------------------
m0, m1 = ICM(
    X_E=data_exp[:,1], X_O=data_obs[:,1], 
    T_E=data_exp[:,2], T_O=data_obs[:,2], 
    Y_E=data_exp[:,3], Y_O=data_obs[:,3],  
    r=2, ID=1, AD=0, rho=best_rho
)

# Predictions on test data
test_data = np.arange(np.min(np.r_[data_exp[:,1], data_obs[:,1]]),
                      np.max(np.r_[data_exp[:,1], data_obs[:,1]]), 0.01)

predY0_exp = m0.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])
predY0_obs = m0.predict_noiseless(np.c_[np.vstack(test_data), np.ones(test_data.shape[0])])
predY1_exp = m1.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])
predY1_obs = m1.predict_noiseless(np.c_[np.vstack(test_data), np.ones(test_data.shape[0])])

predCATE_exp = predY1_exp[0] - predY0_exp[0]
varCATE_exp = predY1_exp[1] + predY0_exp[1]
predCATE_obs = predY1_obs[0] - predY0_obs[0]
varCATE_obs = predY1_obs[1] + predY0_obs[1]

# Ground truth
true_cate = 1 + test_data

# Compute RMSE
from sklearn.metrics import mean_squared_error
print("RMSE - ICMexp: ", mean_squared_error(predCATE_exp, true_cate, squared=False))
print("RMSE - ICMobs: ", mean_squared_error(predCATE_obs, true_cate, squared=False))
